In [ ]:
# Scenario: AI System for Detecting Lung Diseases from Chest X-rays
# 🚨 The Problem

# A large hospital receives thousands of chest X-rays daily.

# Radiologists are:

# Overworked

# Limited in number

# Required to make fast decisions

# Sometimes critical conditions like:

# 👉 Pneumonia
# 👉 COVID-19
# 👉 Normal lungs

# must be identified within minutes.

# Delays can cost lives.

# 💡 The Solution: Build an AI Assistant

# The hospital decides to deploy an AI model that can pre-screen X-rays and
# alert doctors.

# But there is a challenge…

# ❗ Medical datasets are usually SMALL.

# Training a deep neural network from scratch would require:

# Millions of labeled X-rays

# Massive GPU clusters

# Months of training

# Not practical.

# ⭐ Enter Transfer Learning (Your Code)

# Instead of starting from zero, engineers use:

# 👉 ResNet50 trained on ImageNet

# Although ImageNet contains everyday objects (dogs, cars, etc.),
# the early CNN layers learn universal visual patterns, like:

# ✅ Edges
# ✅ Gradients
# ✅ Textures
# ✅ Shapes

# These features are also present in medical scans.

# Import PyTorch library (main deep learning framework)
import torch

# Import torchvision models (contains pre-trained CNN architectures)
import torchvision.models as models

# Import neural network module from PyTorch
from torch import nn


# -------------------------------------------------------
# Step 1: Load a Pre-trained ResNet50 Model
# -------------------------------------------------------

# ResNet50 is a deep convolutional neural network with 50 layers.
# It is already trained on the ImageNet dataset (1.2 million images, 1000 classes).
# Using a pre-trained model allows us to reuse learned visual features
# such as edges, textures, shapes, and patterns.

model = models.resnet50(pretrained=True)


# -------------------------------------------------------
# Step 2: Freeze All Pre-trained Layers
# -------------------------------------------------------

# Transfer learning strategy:
# We freeze earlier layers so their weights do not change during training.
# These layers already learned general image features.

for param in model.parameters():
    param.requires_grad = False

# requires_grad = False means:
# - Gradients will NOT be computed
# - Weights will NOT be updated during backpropagation
# - Training becomes faster and needs less data


# -------------------------------------------------------
# Step 3: Modify the Final Classification Layer
# -------------------------------------------------------

# The original ResNet50 final layer predicts 1000 classes (ImageNet).
# Our problem only has 3 classes:
# 1. Normal
# 2. Pneumonia
# 3. COVID-19

num_classes = 3

# Replace the last fully connected layer (fc)
# model.fc.in_features gives the number of input features to the layer
# (2048 for ResNet50)

model.fc = nn.Linear(model.fc.in_features, num_classes)

# Now the model output will be 3 neurons instead of 1000.


# -------------------------------------------------------
# Step 4: Train Only the New Layer
# -------------------------------------------------------

# Since earlier layers are frozen,
# only the new fully connected layer will be trained.

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable params: {trainable_params}")


# -------------------------------------------------------
# Expected Output Explanation
# -------------------------------------------------------

# The output will show the number of parameters that are trainable.
# Because we froze the backbone network, only the new classification
# layer parameters will be updated during training.

# This is a typical Transfer Learning workflow used in:
# - Medical Image Analysis
# - Face Recognition
# - Object Detection
# - Small dataset problems

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 143MB/s]


Trainable params: 6147


In [1]:
# Scenario: AI System for Detecting Fraud in Financial Transactions
# 🚨 The Problem
# A major bank processes millions of transactions daily.
# Fraud analysts are:
# - Overwhelmed by the sheer volume
# - Limited in number
# - Required to make instant decisions
# - Missing subtle fraud patterns hidden in massive datasets
# Critical cases like:
# 👉 Credit card fraud
# 👉 Money laundering
# 👉 Normal transactions
# must be flagged in real-time.
# Delays can cost millions and damage customer trust.

# 💡 The Solution: Build an AI Assistant
# The bank deploys an AI model that can pre-screen transactions and alert analysts.
# But here’s the challenge…
# ❗ Fraud datasets are usually imbalanced and small.
# Training a deep neural network from scratch would require:
# - Billions of labeled transactions
# - Huge compute clusters
# - Months of training
# Not practical.

# ⭐ Enter Transfer Learning (Your Code)
# Instead of starting from zero, engineers use:
# 👉 BERT (Bidirectional Encoder Representations from Transformers) trained on massive text corpora.
# Although BERT was trained on general language (Wikipedia, books, etc.), its early layers learn universal text patterns, like:
# - ✅ Word embeddings
# - ✅ Sentence structures
# - ✅ Semantic relationships
# - ✅ Contextual meaning
# These features are also present in transaction descriptions, merchant details, and customer behavior logs.
# By fine-tuning BERT on the bank’s fraud dataset, the AI can quickly learn to distinguish:
# - Suspicious vs. normal transactions
# - Fraud rings vs. genuine customer activity


 # ============================================================
# BERT FRAUD DETECTION - COMPLETE TRAINING PIPELINE
# ============================================================

import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from sklearn.metrics import classification_report

# -----------------------------
# Step 1: Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------
# Step 2: Sample Dataset (replace with real data)
# -----------------------------
texts = [
    "Payment to unknown merchant at midnight",
    "Unusual transaction detected overseas",
    "Grocery shopping at local store",
    "Electricity bill payment",
    "Suspicious transfer to foreign account",
    "Online purchase from trusted site"
]

labels = [1, 1, 0, 0, 1, 0]  # 1 = Fraud, 0 = Normal


# -----------------------------
# Step 3: Dataset Class
# -----------------------------
class FraudDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx])
        }


# -----------------------------
# Step 4: Load Tokenizer & Model
# -----------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.to(device)


# -----------------------------
# Step 5: DataLoader
# -----------------------------
dataset = FraudDataset(texts, labels, tokenizer)
loader = DataLoader(dataset, batch_size=2, shuffle=True)


# -----------------------------
# Step 6: Loss (Imbalance Handling)
# -----------------------------
class_weights = torch.tensor([1.0, 2.0]).to(device)  # fraud weight higher
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)


# -----------------------------
# Step 7: Optimizer
# -----------------------------
optimizer = optim.AdamW(model.parameters(), lr=2e-5)


# -----------------------------
# Step 8: Training Loop
# -----------------------------
epochs = 50

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        loss = criterion(logits, labels_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


# -----------------------------
# Step 9: Evaluation
# -----------------------------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.cpu().numpy())

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))


# -----------------------------
# Step 10: Test Prediction
# -----------------------------
test_text = ["Unknown international transfer detected"]

inputs = tokenizer(test_text, return_tensors="pt", truncation=True, padding=True)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1)

print("\nTest Prediction:", pred.item())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1, Loss: 2.4426
Epoch 2, Loss: 2.0998
Epoch 3, Loss: 1.6872
Epoch 4, Loss: 1.4800
Epoch 5, Loss: 1.2388
Epoch 6, Loss: 1.2256
Epoch 7, Loss: 1.1123
Epoch 8, Loss: 0.9245
Epoch 9, Loss: 0.7447
Epoch 10, Loss: 0.6074
Epoch 11, Loss: 0.5146
Epoch 12, Loss: 0.4962
Epoch 13, Loss: 0.3784
Epoch 14, Loss: 0.3911
Epoch 15, Loss: 0.2875
Epoch 16, Loss: 0.2279
Epoch 17, Loss: 0.1971
Epoch 18, Loss: 0.1826
Epoch 19, Loss: 0.1510
Epoch 20, Loss: 0.1160
Epoch 21, Loss: 0.1163
Epoch 22, Loss: 0.1234
Epoch 23, Loss: 0.1095
Epoch 24, Loss: 0.0840
Epoch 25, Loss: 0.0791
Epoch 26, Loss: 0.0661
Epoch 27, Loss: 0.0595
Epoch 28, Loss: 0.0741
Epoch 29, Loss: 0.0621
Epoch 30, Loss: 0.0551
Epoch 31, Loss: 0.0470
Epoch 32, Loss: 0.0622
Epoch 33, Loss: 0.0501
Epoch 34, Loss: 0.0465
Epoch 35, Loss: 0.0366
Epoch 36, Loss: 0.0364
Epoch 37, Loss: 0.0525
Epoch 38, Loss: 0.0340
Epoch 39, Loss: 0.0335
Epoch 40, Loss: 0.0380
Epoch 41, Loss: 0.0305
Epoch 42, Loss: 0.0325
Epoch 43, Loss: 0.0372
Epoch 44, Loss: 0.02

In [9]:
!pip install transformers datasets torch scikit-learn
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 120.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:

# Scenario: AI System for Detecting Fraud in Financial Transactions
# 🚨 The Problem
# A major bank processes millions of transactions daily.
# Fraud analysts are:
# - Overwhelmed by the sheer volume
# - Limited in number
# - Required to make instant decisions
# - Missing subtle fraud patterns hidden in massive datasets
# Critical cases like:
# 👉 Credit card fraud
# 👉 Money laundering
# 👉 Normal transactions
# must be flagged in real-time.
# Delays can cost millions and damage customer trust.

# 💡 The Solution: Build an AI Assistant
# The bank deploys an AI model that can pre-screen transactions and alert analysts.
# But here’s the challenge…
# ❗ Fraud datasets are usually imbalanced and small.
# Training a deep neural network from scratch would require:
# - Billions of labeled transactions
# - Huge compute clusters
# - Months of training
# Not practical.

# !pip install transformers datasets torch scikit-learn
# Install (run once in Colab)
# !pip install transformers datasets torch scikit-learn

import torch
import numpy as np
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ================================
# 1. DEVICE SETUP (VERY IMPORTANT)
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================================
# 2. DATASET
# ================================
data = {
    "text": [
        "Payment to Amazon",
        "Transfer to offshore account",
        "Grocery store purchase",
        "Rapid withdrawals detected",
        "Salary credited",
        "Foreign suspicious transaction",
        "Multiple failed login attempts",
        "Large transfer at midnight",
        "Online shopping payment",
        "ATM withdrawal normal"
    ],
    "label": [0,1,0,1,0,1,1,1,0,0]
}

dataset = Dataset.from_dict(data)

# ================================
# 3. TOKENIZER
# ================================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# ================================
# 4. TRAIN-TEST SPLIT
# ================================
train_test = dataset.train_test_split(test_size=0.3)
train_dataset = train_test["train"]
test_dataset = train_test["test"]

# ================================
# 5. MODEL
# ================================
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.to(device)

# ================================
# 6. TRAINING CONFIG
# ================================
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=5,
    logging_steps=1,
    eval_strategy="no",
    save_strategy="no"
)

# ================================
# 7. TRAINER
# ================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

# ================================
# 8. TRAIN
# ================================
trainer.train()

# ================================
# 9. EVALUATION
# ================================
predictions = trainer.predict(test_dataset)

logits = predictions.predictions
labels = predictions.label_ids
preds = np.argmax(logits, axis=1)

print("\nManual Evaluation:")
print("Accuracy:", accuracy_score(labels, preds))
print("Precision:", precision_score(labels, preds, zero_division=0))
print("Recall:", recall_score(labels, preds, zero_division=0))
print("F1 Score:", f1_score(labels, preds, zero_division=0))

# ================================
# 10. PREDICTION FUNCTION (FIXED)
# ================================
def predict(text):
    model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    # 🔥 CRITICAL FIX (device alignment)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()

    return "Fraud" if pred == 1 else "Normal"

# ================================
# 11. TEST
# ================================
print("\nPredictions:")
print("1.", predict("Unusual large transfer to foreign account"))
print("2.", predict("Paid electricity bill"))

Using device: cuda


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
1,0.581903
2,0.613258
3,0.474844
4,0.638561
5,0.555243
6,0.383995
7,0.489729
8,0.431792
9,0.376447
10,0.395820



Manual Evaluation:
Accuracy: 0.6666666666666666
Precision: 1.0
Recall: 0.5
F1 Score: 0.6666666666666666

Predictions:
1. Fraud
2. Normal


In [3]:
#  The Solution: Build an AI Assistant
# The university deploys an AI model that can pre-screen essays and alert professors about suspicious content.
# But here’s the challenge…
# ❗ Academic plagiarism datasets are usually small and domain-specific.
# Training a deep NLP model from scratch would require:
# - Millions of labeled essays
# - Huge compute resources
# - Months of training
# Not practical.

# ⭐ Enter Transfer Learning (Your Code)
# Instead of starting from zero, engineers use:
# 👉 GPT-style language models trained on massive text corpora (books, articles, Wikipedia).
# Although these corpora contain general language, the early layers learn universal text features, like:
# - ✅ Grammar patterns
# - ✅ Semantic meaning
# - ✅ Sentence structures
# - ✅ Contextual relationships
# These features are also present in student essays.
# By fine-tuning the model on a smaller plagiarism dataset, the AI can quickly learn to distinguish:
# - Original vs. copied content
# - Paraphrased vs. genuinely written text
# - Citation misuse vs. proper referencing

# ⚡ Just like ResNet50 helps radiologists with X-rays, and BERT helps fraud analysts with transactions, GPT-style models help
# educators with essays — all leveraging transfer learning to solve problems where data is scarce but accuracy is critical.


 # !pip install transformers datasets torch scikit-learn

import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ================================
# 1. DEVICE
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# ================================
# 2. DATASET (Plagiarism)
# ================================
data = {
    "text": [
        "This essay is completely original work",
        "Copied from Wikipedia without citation",
        "Student wrote this in their own words",
        "This paragraph is taken from an online article",
        "Properly referenced academic writing",
        "Paraphrased content from a blog",
        "Unique analysis by student",
        "Direct copy paste detected",
        "Well written original essay",
        "Suspicious similarity with source"
    ],
    "label": [0,1,0,1,0,1,0,1,0,1]
}

dataset = Dataset.from_dict(data)

# ================================
# 3. TOKENIZER
# ================================
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# ================================
# 4. SPLIT
# ================================
split = dataset.train_test_split(test_size=0.3)
train_dataset = split["train"]
test_dataset = split["test"]

# ================================
# 5. MODEL (TRANSFER LEARNING)
# ================================
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.to(device)

# ================================
# 6. 🔥 FREEZE LAYERS
# ================================
for param in model.distilbert.embeddings.parameters():
    param.requires_grad = False

# Freeze first 4 layers (DistilBERT has 6 layers total)
for layer in model.distilbert.transformer.layer[:4]:
    for param in layer.parameters():
        param.requires_grad = False

print("Layer freezing done!")

# ================================
# 7. TRAINING CONFIG
# ================================
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=5,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no"
)

# ================================
# 8. TRAINER
# ================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

# ================================
# 9. TRAIN
# ================================
trainer.train()

# ================================
# 10. EVALUATION
# ================================
predictions = trainer.predict(test_dataset)

logits = predictions.predictions
labels = predictions.label_ids
preds = np.argmax(logits, axis=1)

print("\nEvaluation:")
print("Accuracy:", accuracy_score(labels, preds))
print("Precision:", precision_score(labels, preds, zero_division=0))
print("Recall:", recall_score(labels, preds, zero_division=0))
print("F1:", f1_score(labels, preds, zero_division=0))

# ================================
# 11. PREDICTION FUNCTION
# ================================
def predict(text):
    model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()

    return "Plagiarized" if pred == 1 else "Original"

# ================================
# 12. TEST
# ================================
print("\nPredictions:")
print("1.", predict("This paragraph is copied from an article"))
print("2.", predict("This is my own analysis and writing"))

Using: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Layer freezing done!


Step,Training Loss
1,0.714592
2,0.728651
3,0.687980
4,0.646177
5,0.677031
6,0.651072
7,0.679611
8,0.672910
9,0.672711
10,0.652790



Evaluation:
Accuracy: 0.3333333333333333
Precision: 0.3333333333333333
Recall: 1.0
F1: 0.5

Predictions:
1. Plagiarized
2. Plagiarized


In [5]:
# Install required libraries if not already installed:
# pip install transformers datasets peft accelerate

# Install required libraries if not already installed:
# pip install transformers datasets peft accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
from peft import get_peft_model, LoraConfig

# 1. Load base model and tokenizer
model_name = "gpt2"   # you can replace with another causal LM
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure pad token exists (GPT-2 doesn't have one by default)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Define LoRA configuration
lora_config = LoraConfig(
    r=8,                  # rank of low-rank matrices
    lora_alpha=32,        # scaling factor
    lora_dropout=0.1,     # dropout for regularization
    bias="none",          # no bias terms updated
    task_type="CAUSAL_LM",# language modeling task
    target_modules=["c_attn"]  # GPT-2 attention projection layers
)

# Wrap model with LoRA adapters
model = get_peft_model(model, lora_config)

# 3. Create a tiny synthetic dataset
train_texts = ["Hello world", "LoRA fine-tuning is fun", "Transformers are powerful"]
eval_texts = ["Testing evaluation", "Another sample"]

train_dataset = Dataset.from_dict({"text": train_texts})
eval_dataset = Dataset.from_dict({"text": eval_texts})

# Tokenization function with labels
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )
    tokens["labels"] = tokens["input_ids"].copy()  # labels required for loss
    return tokens

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 4. Training configuration
training_args = TrainingArguments(
    output_dir="./lora-finetuned-model",   # save directory
    per_device_train_batch_size=2,         # small batch size
    gradient_accumulation_steps=2,         # accumulate gradients
    learning_rate=2e-4,                    # tuned for LoRA
    num_train_epochs=10,                    # demo run
    logging_steps=1,                       # log every step
    save_strategy="epoch",                 # save at end of epoch
    fp16=True                              # mixed precision training
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

# 6. Train
trainer.train()

# 7. Save only LoRA adapter weights (few MB instead of GBs!)
model.save_pretrained("./lora-weights")

print("Training complete. LoRA weights saved in ./lora-weights")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Step,Training Loss
1,8.163726
2,8.021153
3,8.017034
4,7.883471
5,7.611611
6,7.511741
7,7.594941
8,7.197976
9,7.476475
10,7.624592


Training complete. LoRA weights saved in ./lora-weights


In [6]:
# Big Idea Behind This Code

# Instead of shipping a 500MB–10GB model every time…

# You ship:

# 👉 A tiny adapter (few MB)

# Then dynamically attach it to the base model.

# This enables:

# ✅ Modular AI systems
# ✅ Fast deployment
# ✅ Cheap storage
# ✅ Multi-domain adapters

# This pattern is exploding across the industry.

# 🚀 Real-World Scenario (VERY Powerful)
# 🛒 Scenario: E-Commerce Company Builds Multiple AI Assistants

# An e-commerce giant wants different AI behaviors:

# Sentiment analyzer

# Product recommender

# Customer support bot

# Return-policy assistant

# But retraining separate LLMs would cost millions.

# So engineers do something smart:

# 👉 One Base Model
# 👉 Multiple LoRA Adapters


# Install required libraries if not already installed:
# pip install transformers datasets peft accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
from peft import get_peft_model, LoraConfig, PeftModel

# 1. Load base model and tokenizer
model_name = "gpt2"   # you can replace with another causal LM
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Define LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn"]  # GPT-2 uses 'c_attn' for QKV projection
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)

# 3. Create a tiny synthetic dataset
train_texts = ["Hello world", "LoRA fine-tuning is fun", "Transformers are powerful"]
eval_texts = ["Testing evaluation", "Another sample"]

train_dataset = Dataset.from_dict({"text": train_texts})
eval_dataset = Dataset.from_dict({"text": eval_texts})

# Tokenization function with labels
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )
    tokens["labels"] = tokens["input_ids"].copy()  # labels required for loss
    return tokens

train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 4. Training configuration
training_args = TrainingArguments(
    output_dir="./lora-finetuned-model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=1,   # keep small for demo
    logging_steps=1,
    save_strategy="epoch",
    fp16=True
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

# 6. Train
trainer.train()

# 7. Save only LoRA adapter weights (few MB instead of GBs!)
model.save_pretrained("lora-sentiment")  # folder will contain adapter_config.json

print("Training complete. LoRA weights saved in ./lora-sentiment")

# 8. Reload base model and attach adapter for inference
base_model = AutoModelForCausalLM.from_pretrained("gpt2")
sentiment_model = PeftModel.from_pretrained(base_model, "lora-sentiment")

# Example inference
input_text = "I love using transformers and LoRA adapters!"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids
output_ids = sentiment_model.generate(input_ids, max_length=50)
print("Output:", tokenizer.decode(output_ids[0], skip_special_tokens=True))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Step,Training Loss
1,8.163726


Training complete. LoRA weights saved in ./lora-sentiment


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Output: I love using transformers and LoRA adapters!

I love using transformers and LoRA adapters! I love using transformers and LoRA adapters! I love using transformers and LoRA adapters! I love using transformers and LoRA


In [3]:
# Scenario: Fraud Detection in Customer Support Chats
# - Problem: Fraudsters often contact bank support pretending to be customers, trying to reset passwords or gain access.
# - Challenge: Analysts can’t manually read millions of chat transcripts.
# - Solution: Use LoRA fine‑tuning on a pretrained language model (like distilbert-base-uncased) to classify chats as:
# - 0 → Normal inquiry
# - 1 → Fraud attempt
# - 2 → Suspicious but unclear

# pip install transformers datasets peft accelerate scikit-learn

import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = {
    "text": [
        "Hi, I want to check my account balance",
        "Please reset my password urgently",
        "I forgot my password and need access",
        "Transfer all funds to this new account now",
        "What are your bank timings?",
        "My OTP is not working",
        "I need to update my phone number",
        "Send money immediately to avoid penalty",
        "How can I apply for a loan?",
        "Give me access to account without OTP"
    ],
    "label": [0,1,2,1,0,0,0,1,0,2]
}

dataset = Dataset.from_dict(data)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

split = dataset.train_test_split(test_size=0.3)
train_dataset = split["train"]
test_dataset = split["test"]

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=["q_lin", "v_lin"]
)

model = get_peft_model(model, lora_config)
model.to(device)

training_args = TrainingArguments(
    output_dir="./fraud-chat-model",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    num_train_epochs=5,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0)
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results = trainer.evaluate()
print(results)

def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    mapping = {
        0: "Normal Inquiry",
        1: "Fraud Attempt",
        2: "Suspicious"
    }
    return mapping[pred]

print(predict("I forgot my password please reset urgently"))
print(predict("What is my account balance"))
print(predict("Give access without OTP"))

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.055600
2,1.151296
3,1.099190
4,1.064919
5,1.119790
6,0.938022
7,1.009518
8,1.045238
9,1.025939
10,0.946119


{'eval_loss': 1.0934447050094604, 'eval_accuracy': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0, 'eval_runtime': 2.6904, 'eval_samples_per_second': 1.115, 'eval_steps_per_second': 0.372, 'epoch': 5.0}
Fraud Attempt
Fraud Attempt
Fraud Attempt
